# Parameter Tuning Experiments
- ###### Compare temperature settings (0.0, 0.7, 1.5)
- ###### Measure creativity vs consistency trade-offs

Below is a **single Python file** that compares **temperature settings (0.0, 0.7, 1.5)** using the **OpenAI Python SDK**. It sends the same prompt multiple times for each temperature, measures response consistency, response length, and lexical diversity (a simple creativity proxy), then prints a comparison report.


### Expected observations

| Temperature | Creativity | Consistency | Typical Use                                   |
| ----------- | ---------- | ----------- | --------------------------------------------- |
| **0.0**     | Low        | Very High   | Coding, factual Q&A, data extraction          |
| **0.7**     | Medium     | Medium      | Chatbots, content writing                     |
| **1.5**     | High       | Low         | Brainstorming, storytelling, creative writing |

This experiment demonstrates the **creativity vs. consistency trade-off** by generating multiple responses for the same prompt and comparing:

* **Average response length**
* **Lexical diversity** (higher suggests more varied wording)
* **Jaccard similarity** between runs (higher indicates greater consistency)

## Example 1


In [3]:
"""
LLM Parameter Tuning Experiment
Compare Temperature Settings: 0.0, 0.7, 1.5

Requirements:
    pip install openai

Set API Key:
    export OPENAI_API_KEY="your_api_key"
"""

import os
import re
from collections import Counter
from statistics import mean

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -----------------------------
# Configuration
# -----------------------------
MODEL = "gpt-4.1-mini"      # Change to any supported model
TEMPERATURES = [0.0, 0.7, 1.5]
RUNS_PER_TEMP = 5

PROMPT = """
Write a short paragraph (80-100 words) describing the future of artificial intelligence.
"""


# -----------------------------
# Helper Functions
# -----------------------------
def get_response(temp):
    response = client.responses.create(
        model=MODEL,
        input=PROMPT,
        temperature=temp,
    )
    return response.output_text.strip()


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


def lexical_diversity(text):
    words = tokenize(text)
    if not words:
        return 0
    return len(set(words)) / len(words)


def similarity(a, b):
    """
    Simple Jaccard similarity between word sets.
    Higher value = more consistent.
    """
    wa = set(tokenize(a))
    wb = set(tokenize(b))

    if not wa or not wb:
        return 0

    return len(wa & wb) / len(wa | wb)


# -----------------------------
# Experiment
# -----------------------------
results = {}

print("=" * 70)
print("LLM Temperature Experiment")
print("=" * 70)

for temp in TEMPERATURES:

    print(f"\nTemperature = {temp}")
    print("-" * 70)

    outputs = []

    for i in range(RUNS_PER_TEMP):
        text = get_response(temp)
        outputs.append(text)

        print(f"\nRun {i+1}:")
        print(text)
        print()

    lengths = [len(tokenize(x)) for x in outputs]
    diversity = [lexical_diversity(x) for x in outputs]

    similarities = []
    for i in range(len(outputs)):
        for j in range(i + 1, len(outputs)):
            similarities.append(similarity(outputs[i], outputs[j]))

    results[temp] = {
        "avg_length": mean(lengths),
        "avg_diversity": mean(diversity),
        "avg_similarity": mean(similarities)
    }

# -----------------------------
# Summary
# -----------------------------
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    f"{'Temp':<10}"
    f"{'Avg Length':<15}"
    f"{'Lexical Diversity':<22}"
    f"{'Consistency':<15}"
)

for temp in TEMPERATURES:
    r = results[temp]
    print(
        f"{temp:<10}"
        f"{r['avg_length']:<15.2f}"
        f"{r['avg_diversity']:<22.3f}"
        f"{r['avg_similarity']:<15.3f}"
    )

print("\nInterpretation")
print("-" * 70)
print("Temperature 0.0 :")
print("  • Highest consistency")
print("  • Lowest creativity")
print("  • Best for factual tasks and coding\n")

print("Temperature 0.7 :")
print("  • Balanced creativity and consistency")
print("  • Good for general-purpose applications\n")

print("Temperature 1.5 :")
print("  • Most creative")
print("  • Lowest consistency")
print("  • Suitable for storytelling and brainstorming")

LLM Temperature Experiment

Temperature = 0.0
----------------------------------------------------------------------

Run 1:
The future of artificial intelligence (AI) promises transformative advancements across various sectors, from healthcare and education to transportation and entertainment. As AI systems become more sophisticated, they will enhance decision-making, automate complex tasks, and foster innovation. Ethical considerations and regulations will play a crucial role in ensuring responsible development and deployment. With continued progress in machine learning, natural language processing, and robotics, AI is expected to create smarter, more adaptive technologies that improve quality of life, drive economic growth, and address global challenges such as climate change and disease management. The future of AI holds immense potential, balanced by the need for careful stewardship.


Run 2:
The future of artificial intelligence (AI) promises transformative advancements across va

## Example 2

Here's a script that runs the same set of prompts across temperatures 0.0, 0.7, and 1.5, with multiple trials per setting so you can quantify both sides of the trade-off:

- **Consistency**: average pairwise TF-IDF cosine similarity across repeated trials at the same temperature (closer to 1.0 = more repeatable output)
- **Creativity**: a blend of lexical diversity (unique/total words per response) and inter-trial novelty (1 − consistency)

It saves raw responses to `temperature_experiment_results.json` and an aggregated table to `temperature_experiment_summary.csv`, plus prints a summary table to console.

To run it: set `OPENAI_API_KEY`, `pip install openai scikit-learn`, then `python temperature_experiment.py`. You can swap `MODEL`, add more `PROMPTS`, or bump `TRIALS_PER_SETTING` for more statistical stability (costs more API calls, though).

In [4]:
"""
Temperature Parameter Tuning Experiment
=========================================
Compares OpenAI chat completion outputs across temperature settings
(0.0, 0.7, 1.5) and measures the creativity vs. consistency trade-off.

Setup:
    pip install openai scikit-learn numpy
    export OPENAI_API_KEY="sk-..."

Run:
    python temperature_experiment.py

Output:
    - Console summary table
    - temperature_experiment_results.json (raw responses + metrics)
    - temperature_experiment_summary.csv (aggregated metrics per temperature)
"""

import os
import csv
import json
import itertools
import statistics
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

MODEL = "gpt-4o-mini"          # swap for any chat-completion-capable model
TEMPERATURES = [0.0, 0.7, 1.5]
TRIALS_PER_SETTING = 5          # repeats per (prompt, temperature) pair -> consistency signal

PROMPTS = [
    "Write a one-sentence description of a sunset.",
    "Complete this story opener: 'The old lighthouse keeper had a secret...'",
    "List three unusual uses for a paperclip.",
]

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# ---------------------------------------------------------------------------
# API call
# ---------------------------------------------------------------------------

def call_model(prompt: str, temperature: float) -> str:
    """Single chat completion call at a given temperature."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=150,
    )
    return response.choices[0].message.content.strip()


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------

def lexical_diversity(text: str) -> float:
    """Type-token ratio: unique words / total words. Higher = more varied vocabulary."""
    words = text.lower().split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)


def avg_response_length(texts: list[str]) -> float:
    return statistics.mean(len(t.split()) for t in texts)


def consistency_score(texts: list[str]) -> float:
    """
    Average pairwise cosine similarity (TF-IDF) across repeated trials
    for the same prompt/temperature. 1.0 = identical outputs every time,
    0.0 = completely different outputs. High score = high consistency.
    """
    if len(texts) < 2:
        return 1.0
    vectorizer = TfidfVectorizer().fit_transform(texts)
    sims = cosine_similarity(vectorizer)
    n = len(texts)
    pairs = [sims[i][j] for i, j in itertools.combinations(range(n), 2)]
    return statistics.mean(pairs)


def creativity_score(texts: list[str]) -> float:
    """
    Blend of lexical diversity (within each response) and inter-trial
    novelty (1 - consistency). Higher = more creative/varied output.
    """
    diversity = statistics.mean(lexical_diversity(t) for t in texts)
    novelty = 1 - consistency_score(texts)
    return statistics.mean([diversity, novelty])


# ---------------------------------------------------------------------------
# Experiment runner
# ---------------------------------------------------------------------------

def run_experiment():
    results = []  # raw record per prompt/temperature/trial
    per_setting = {}  # (prompt, temp) -> list of response texts

    for prompt in PROMPTS:
        for temp in TEMPERATURES:
            texts = []
            for trial in range(TRIALS_PER_SETTING):
                text = call_model(prompt, temp)
                texts.append(text)
                results.append({
                    "prompt": prompt,
                    "temperature": temp,
                    "trial": trial,
                    "response": text,
                })
                print(f"[T={temp}] trial {trial+1}/{TRIALS_PER_SETTING} -> {text[:60]}...")
            per_setting[(prompt, temp)] = texts

    # Aggregate metrics per temperature (averaged across all prompts)
    summary = {}
    for temp in TEMPERATURES:
        cons_scores, creat_scores, lengths = [], [], []
        for prompt in PROMPTS:
            texts = per_setting[(prompt, temp)]
            cons_scores.append(consistency_score(texts))
            creat_scores.append(creativity_score(texts))
            lengths.append(avg_response_length(texts))
        summary[temp] = {
            "avg_consistency": round(statistics.mean(cons_scores), 4),
            "avg_creativity": round(statistics.mean(creat_scores), 4),
            "avg_response_length_words": round(statistics.mean(lengths), 2),
        }

    return results, summary


def print_summary(summary: dict):
    print("\n" + "=" * 60)
    print("TEMPERATURE TRADE-OFF SUMMARY")
    print("=" * 60)
    print(f"{'Temp':<8}{'Consistency':<14}{'Creativity':<14}{'Avg Words'}")
    for temp, m in summary.items():
        print(f"{temp:<8}{m['avg_consistency']:<14}{m['avg_creativity']:<14}{m['avg_response_length_words']}")
    print("=" * 60)
    print("Consistency: 1.0 = identical repeated outputs, 0.0 = fully divergent")
    print("Creativity:  higher = more lexical/inter-trial variation")


def save_outputs(results: list, summary: dict):
    with open("temperature_experiment_results.json", "w") as f:
        json.dump({"raw_results": results, "summary": summary}, f, indent=2)

    with open("temperature_experiment_summary.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["temperature", "avg_consistency", "avg_creativity", "avg_response_length_words"])
        for temp, m in summary.items():
            writer.writerow([temp, m["avg_consistency"], m["avg_creativity"], m["avg_response_length_words"]])

    print("\nSaved: temperature_experiment_results.json, temperature_experiment_summary.csv")


if __name__ == "__main__":
    raw_results, summary_stats = run_experiment()
    print_summary(summary_stats)
    save_outputs(raw_results, summary_stats)

[T=0.0] trial 1/5 -> The sun dipped below the horizon, painting the sky in vibran...
[T=0.0] trial 2/5 -> The sun dipped below the horizon, painting the sky in vibran...
[T=0.0] trial 3/5 -> The sun dipped below the horizon, painting the sky in vibran...
[T=0.0] trial 4/5 -> The sun dipped below the horizon, painting the sky in vibran...
[T=0.0] trial 5/5 -> The sun dipped below the horizon, painting the sky in vibran...
[T=0.7] trial 1/5 -> As the sun dips below the horizon, the sky transforms into a...
[T=0.7] trial 2/5 -> The sun dipped below the horizon, painting the sky in vibran...
[T=0.7] trial 3/5 -> The sunset painted the sky in a breathtaking tapestry of war...
[T=0.7] trial 4/5 -> As the sun dips below the horizon, the sky ignites in a brea...
[T=0.7] trial 5/5 -> As the sun dips below the horizon, the sky transforms into a...
[T=1.5] trial 1/5 -> The sun sinks gently into the horizon, painting the sky with...
[T=1.5] trial 2/5 -> As the day transitions into twilight, the su